# Day 2: Neutrino Oscillation Spectra

Today you will build the plot particle physicists use to see neutrino oscillation: a near-detector energy spectrum compared with a far-detector spectrum.

By the end you should be able to:
- explain why a far-detector spectrum can have a dip,
- make an overlay plot and a far/near ratio plot,
- use `nuosclab` to connect that dip to oscillation parameters,
- describe how toy statistical fluctuations change what the plot looks like.

Prerequisites: running notebook cells, reading a histogram, and basic Python variables.


## 1. Load the tools

`nuosclab` calculates the oscillation probabilities. We will wrap it with a toy beam model so the output looks like binned detector data.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from nuosclab import ExplorerConfig, PMNSParams, compute_curves
from nuosclab.presets import PRESETS

plt.style.use("seaborn-v0_8-whitegrid")

### Setup check

`nuosclab` is the package that actually computes the oscillation physics behind the sliders below. You do not install it by hand — running `uv sync` in this repository (see the README) already pulled it in as a dependency, straight from its GitHub repository. If the import below fails, the most likely fix is to make sure the notebook's kernel (top-right of the window) says **EPIC Curriculum**, not "Python 3 (ipykernel)" or similar.

!!! `nuosclab` is a new, actively developed research package (currently `v0.8.0`, pre-1.0) written for exactly this kind of teaching and validation use. Its API can still change between versions, and you may notice rough edges. That is normal for a young scientific package, not a sign you did something wrong.


In [ ]:
from importlib.metadata import version

print("nuosclab version:", version("nuosclab"))

### A direct nuosclab call, before we wrap it

Everything below in this notebook goes through helper functions (`pmns_with`, `oscillation_curves`, `toy_spectra`) that wrap `nuosclab` so the sliders are simple to use. But it helps to see the raw call underneath at least once.

The three pieces:

- **`PMNSParams`** holds the mixing angles and mass-squared differences that describe standard three-flavor oscillation. Leaving it empty (`PMNSParams()`) uses PDG best-fit defaults.
- **`ExplorerConfig`** bundles which `experiment` preset to use (baseline, density, energy range — see `PRESETS["NOvA"]`) together with the physics parameters.
- **`compute_curves(config)`** runs the calculation and returns an `ExplorerCurves` object with an `energy_gev` array and a `live` array of oscillation probabilities.

`live` has shape `(n_points, 3, 3)` — for each energy point, a 3×3 grid of `P[beta, alpha]` (probability a neutrino produced in flavor `alpha` is measured in flavor `beta`), indexed `0=e, 1=μ, 2=τ`. So `live[:, 1, 1]` is muon-neutrino *survival* and `live[:, 0, 1]` is muon-to-electron *appearance* — the two curves used throughout this notebook.


In [ ]:
demo_config = ExplorerConfig(experiment="NOvA", pmns=PMNSParams())
demo_curves = compute_curves(demo_config)

print("energy grid shape:", demo_curves.energy_gev.shape)
print("probability array shape:", demo_curves.live.shape)

survival_at_first_point = demo_curves.live[0, 1, 1]
appearance_at_first_point = demo_curves.live[0, 0, 1]
print("P(nu_mu -> nu_mu) at the first energy point:", survival_at_first_point)
print("P(nu_mu -> nu_e) at the first energy point:", appearance_at_first_point)

The helper functions below make exactly this call, just with tutorial-friendly arguments (`theta23_deg`, `dm32`, `delta_cp_deg`) instead of raw `PMNSParams` objects, and they slice `live[:, 1, 1]` / `live[:, 0, 1]` for you.


In [ ]:
DEFAULT_PMNS = PMNSParams()


def pmns_with(theta23_deg=49.2, dm32=2.441e-3, delta_cp_deg=-91.7):
    """Build PMNS parameters from tutorial controls."""
    dm21 = DEFAULT_PMNS.dm21
    return PMNSParams(
        th12=DEFAULT_PMNS.th12,
        th13=DEFAULT_PMNS.th13,
        th23=np.radians(theta23_deg),
        dm21=dm21,
        dm31=dm32 + dm21,
        delta_cp=np.radians(delta_cp_deg),
    )


def oscillation_curves(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32=2.441e-3,
    delta_cp_deg=-91.7,
    n_points=700,
):
    """Compute nu_mu disappearance and nu_e appearance with nuosclab."""
    return compute_curves(
        ExplorerConfig(
            experiment=experiment,
            pmns=pmns_with(theta23_deg, dm32, delta_cp_deg),
            n_points=n_points,
            include_standard=False,
            include_nominal=True,
        )
    )


def beam_shape(energy, peak, width):
    """A smooth toy beam energy spectrum, not an official flux prediction."""
    low_energy_turn_on = 1.0 - np.exp(-((energy / 0.45) ** 2))
    return low_energy_turn_on * np.exp(-0.5 * ((energy - peak) / width) ** 2)


def toy_spectra(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32=2.441e-3,
    delta_cp_deg=-91.7,
    total_near_events=6000,
    n_bins=24,
    seed=7,
    fluctuate=True,
):
    """Return binned near/far toy spectra with optional Poisson fluctuations."""
    preset = PRESETS[experiment]
    edges = np.linspace(*preset.E_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    width = 0.65 if experiment == "NOvA" else 0.9
    if experiment == "T2K":
        width = 0.22

    curves = oscillation_curves(
        experiment=experiment,
        theta23_deg=theta23_deg,
        dm32=dm32,
        delta_cp_deg=delta_cp_deg,
        n_points=900,
    )
    survival = np.interp(centers, curves.energy_gev, curves.live[:, 1, 1])
    appearance = np.interp(centers, curves.energy_gev, curves.live[:, 0, 1])

    near_expect = beam_shape(centers, preset.E_peak, width)
    near_expect = total_near_events * near_expect / near_expect.sum()
    far_expect = near_expect * survival

    rng = np.random.default_rng(seed)
    if fluctuate:
        near = rng.poisson(near_expect)
        far = rng.poisson(far_expect)
    else:
        near = near_expect
        far = far_expect

    ratio = np.divide(
        far,
        near,
        out=np.full_like(far, np.nan, dtype=float),
        where=near > 0,
    )
    return {
        "experiment": experiment,
        "edges": edges,
        "centers": centers,
        "near": near,
        "far": far,
        "near_expect": near_expect,
        "far_expect": far_expect,
        "ratio": ratio,
        "survival": survival,
        "appearance": appearance,
        "curves": curves,
    }

In [ ]:
def plot_near_far(result):
    """Make the main tutorial plot: spectra above, far/near ratio below."""
    centers = result["centers"]
    widths = np.diff(result["edges"])
    experiment = result["experiment"]

    fig, (ax_top, ax_ratio) = plt.subplots(
        2,
        1,
        figsize=(9, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
        constrained_layout=True,
    )
    ax_top.bar(
        centers,
        result["near"],
        width=widths,
        alpha=0.45,
        label="Near detector toy data",
        color="#4c78a8",
        edgecolor="black",
        linewidth=0.5,
    )
    ax_top.bar(
        centers,
        result["far"],
        width=widths,
        alpha=0.55,
        label="Far detector toy data",
        color="#f58518",
        edgecolor="black",
        linewidth=0.5,
    )
    ax_top.plot(
        centers,
        result["far_expect"],
        color="#9c3d10",
        linewidth=2,
        label="Expected far spectrum",
    )
    ax_top.set_ylabel("Events per bin")
    ax_top.set_title(f"{experiment} toy near/far spectra")
    ax_top.legend()

    ax_ratio.scatter(centers, result["ratio"], color="#b279a2", zorder=3)
    ax_ratio.plot(
        centers,
        result["survival"],
        color="black",
        linewidth=2,
        label=r"$P(\nu_\mu \to \nu_\mu)$",
    )
    ax_ratio.set_xlabel("Reconstructed neutrino energy (GeV)")
    ax_ratio.set_ylabel("Far / near")
    ax_ratio.set_ylim(0, 1.15)
    ax_ratio.legend(loc="lower right")
    return fig


def show_spectra(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32_x1000=2.441,
    total_near_events=6000,
    n_bins=24,
    seed=7,
    fluctuate=True,
):
    result = toy_spectra(
        experiment=experiment,
        theta23_deg=theta23_deg,
        dm32=dm32_x1000 * 1e-3,
        total_near_events=total_near_events,
        n_bins=n_bins,
        seed=seed,
        fluctuate=fluctuate,
    )
    plot_near_far(result)
    plt.show()

## 2. Start with a smooth NOvA-like spectrum

The near detector measures the beam before oscillation has much distance to develop. The far detector sees the same beam after 810 km, so the muon-neutrino survival probability has shaped the spectrum.


In [ ]:
smooth_nova = toy_spectra(experiment="NOvA", fluctuate=False)
plot_near_far(smooth_nova)
plt.show()

## 3. Add toy statistical fluctuations

Real data are counts, not smooth curves. If the expected count in a bin is 25, seeing 21 or 30 is normal. The switch below lets you compare the smooth expectation with a fluctuated toy experiment.


In [ ]:
controls = widgets.VBox(
    [
        widgets.HBox(
            [
                widgets.ToggleButtons(
                    options=["NOvA", "DUNE"],
                    value="NOvA",
                    description="Experiment",
                ),
                widgets.Checkbox(value=True, description="Fluctuate counts"),
            ]
        ),
        widgets.HBox(
            [
                widgets.FloatSlider(
                    value=49.2,
                    min=40.0,
                    max=55.0,
                    step=0.2,
                    description="theta23",
                ),
                widgets.FloatSlider(
                    value=2.441,
                    min=2.1,
                    max=2.8,
                    step=0.01,
                    description="dm32 x1000",
                ),
            ]
        ),
        widgets.HBox(
            [
                widgets.IntSlider(
                    value=6000,
                    min=500,
                    max=20000,
                    step=500,
                    description="Near events",
                ),
                widgets.IntSlider(
                    value=24,
                    min=10,
                    max=50,
                    step=2,
                    description="Bins",
                ),
                widgets.IntSlider(
                    value=7,
                    min=1,
                    max=999,
                    step=1,
                    description="Seed",
                ),
            ]
        ),
    ]
)
experiment_widget = controls.children[0].children[0]
fluctuate_widget = controls.children[0].children[1]
theta_widget = controls.children[1].children[0]
dm32_widget = controls.children[1].children[1]
events_widget = controls.children[2].children[0]
bins_widget = controls.children[2].children[1]
seed_widget = controls.children[2].children[2]

interactive_plot = widgets.interactive_output(
    show_spectra,
    {
        "experiment": experiment_widget,
        "theta23_deg": theta_widget,
        "dm32_x1000": dm32_widget,
        "total_near_events": events_widget,
        "n_bins": bins_widget,
        "seed": seed_widget,
        "fluctuate": fluctuate_widget,
    },
)
display(controls, interactive_plot)

## 4. Exercises

1. Turn fluctuations off. Move Δm²₃₂ up and down. What happens to the energy where the dip appears?
2. Turn fluctuations on and reduce the event count. At what point does the dip become hard to see?
3. Switch from NOvA to DUNE. How does the longer baseline change the energy range you need to inspect?


In [ ]:
# Exercise workspace: record one configuration where the dip is easy to see
# and one where it is hard to see. Then run the cell.

easy_to_see = {
    "experiment": "NOvA",
    "near_events": 6000,
    "bins": 24,
    "seed": 7,
}

hard_to_see = {
    "experiment": "NOvA",
    "near_events": 1000,
    "bins": 40,
    "seed": 12,
}

print("Easy configuration:", easy_to_see)
print("Hard configuration:", hard_to_see)

## 5. Reflection

A good final plot is not just technically correct. It should make the physics visible. Before moving on, write one sentence explaining the far/near ratio plot to someone who has never heard of neutrino oscillation.


In [ ]:
my_explanation = "The far detector sees fewer muon neutrinos at some energies because some changed flavor while traveling."
print(my_explanation)